# Week 1 · Burstiness: beyond the average
*MSc Digital Scholarship 2026*

Last time we measured **average sentence length**. But two writers can have exactly the same average
and still sound completely different. One writes steadily, sentence after sentence of similar length.
Another writes a long, winding sentence full of clauses and qualifications, and then stops. Short. Like that.

**Burstiness** is an attempt to put a number on that difference. In this notebook we will:
1. See why an average hides this.
2. Build up a burstiness measure step by step.
3. Apply it to four novels.
4. Ask what it *can't* see, and what people mean when they use the word.

As before: **Shift + Enter** runs a cell, run from the top, and look out for the **Critical check** boxes.

## 0. Setup: tools from last time

Each Colab notebook starts fresh, so this cell recreates the tools we built in the last notebook:
downloading a Gutenberg book, removing the licence text, finding words, and splitting into sentences.
You don't need to read it closely. Just run it.

In [ ]:
import re
import random
import statistics
import urllib.request

import nltk
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


def get_gutenberg(book_id):
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    raw = urllib.request.urlopen(url).read().decode("utf-8")
    return raw.replace("\r\n", "\n")

def strip_gutenberg(raw):
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1
    end = raw.find("*** END OF")
    return raw[start:end].strip()

def words_in(text):
    return re.findall(r"[A-Za-z]+(?:['’-][A-Za-z]+)*", text)

def sentence_lengths(text):
    return [len(words_in(s)) for s in sent_tokenize(text)]

print("Ready.")

## 1. Same average, different feel

Here are three imaginary writers. Each list is the length (in words) of eight sentences in a row.

In [ ]:
metronome = [12, 13, 12, 11, 12, 13, 12, 11]
varied    = [3, 25, 4, 30, 2, 20, 5, 7]
extreme   = [1, 2, 1, 60, 2, 1, 3, 26]

print("metronome:", statistics.mean(metronome))
print("varied:   ", statistics.mean(varied))
print("extreme:  ", statistics.mean(extreme))

All three have an average of exactly 12 words per sentence. If average sentence length were all we measured,
these writers would look identical. Let's draw them:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, name, lengths in zip(axes, ["metronome", "varied", "extreme"], [metronome, varied, extreme]):
    ax.bar(range(1, 9), lengths, color="steelblue")
    ax.axhline(12, color="black", linestyle="--", linewidth=1)   # the average
    ax.set_title(name)
    ax.set_xlabel("sentence number")
axes[0].set_ylabel("words")
plt.show()

> **Discussion.** Can you think of real writers, or genres, that resemble each of these three?
> Would a reader *notice* the difference between them? Would they notice it in a single paragraph, or only over a chapter?

## 2. Measuring spread

### First idea: the range
The simplest measure of spread is the **range**: the longest sentence minus the shortest.
`max` and `min` are functions, like `len`.

In [ ]:
for name, lengths in [("metronome", metronome), ("varied", varied), ("extreme", extreme)]:
    print(name, "range:", max(lengths) - min(lengths))

> **Critical check.** The range depends on just **two** sentences: the longest and the shortest.
> In a whole novel, what would that mean? (Think back to the zero-word "sentence" and the long sentences from the preface in the last notebook.)

### Better idea: the standard deviation
The **standard deviation** uses *every* sentence. Roughly, it is "**how far a typical sentence is from the average**".
- Small standard deviation: sentences cluster close to the average (the metronome).
- Large standard deviation: sentences are often much longer or much shorter than average.

It comes from the `statistics` module. (There are two versions, `stdev` and `pstdev`; we use `pstdev`.
For long texts the difference is tiny.)

In [ ]:
for name, lengths in [("metronome", metronome), ("varied", varied), ("extreme", extreme)]:
    print(name, "standard deviation:", round(statistics.pstdev(lengths), 2))

<details><summary><b>Optional: how is the standard deviation calculated?</b> (click to open)</summary>

1. Work out the average.
2. For each sentence, find how far it is from the average.
3. Square each of those distances (so that "3 shorter" and "3 longer" both count as positive).
4. Average the squares.
5. Take the square root, to get back to "words" as the unit.

The cell below does it by hand for the `varied` writer, so you can check it against `statistics.pstdev`.
</details>

In [ ]:
avg = statistics.mean(varied)
squares = []
for length in varied:
    distance = length - avg
    print(length, "is", distance, "from the average")
    squares.append(distance ** 2)

print("by hand:   ", round(statistics.mean(squares) ** 0.5, 2))
print("statistics:", round(statistics.pstdev(varied), 2))

## 3. A problem of scale

Suppose a fourth writer writes exactly like `varied`, but every sentence is **twice as long**.
Is this writer *burstier*? Or just *wordier*?

In [ ]:
wordy = [2 * x for x in varied]      # double every length
print(wordy)
print("mean:", statistics.mean(wordy), "  standard deviation:", round(statistics.pstdev(wordy), 2))

The standard deviation doubled too. So a writer with longer sentences will tend to get a bigger standard deviation,
even if the *pattern* of long and short is exactly the same.

The usual fix is to divide by the average. This is called the **coefficient of variation**:

In [ ]:
def coefficient_of_variation(lengths):
    return statistics.pstdev(lengths) / statistics.mean(lengths)

print("varied:", round(coefficient_of_variation(varied), 3))
print("wordy: ", round(coefficient_of_variation(wordy), 3))

Now the two are the same, which matches our sense that they have the same *shape*.

## 4. The burstiness index

A widely used **burstiness index** was proposed by the physicists Goh and Barabási (2008), who were studying
the timing of events such as emails being sent, earthquakes and heartbeats. Some events come in bursts, with long quiet gaps in between.

With **σ** (sigma) for the standard deviation and **μ** (mu) for the mean:

$$B = \frac{\sigma - \mu}{\sigma + \mu}$$

It always lies between **−1** and **+1**:
- **−1**: perfectly regular (every value the same).
- **0**: the standard deviation equals the mean. In Goh and Barabási's setting, this is what events happening
  *completely at random* look like.
- **+1**: extremely bursty.

In [ ]:
def burstiness(lengths):
    mu = statistics.mean(lengths)
    sigma = statistics.pstdev(lengths)
    return (sigma - mu) / (sigma + mu)

for name, lengths in [("metronome", metronome), ("varied", varied), ("extreme", extreme), ("wordy", wordy)]:
    print(f"{name:10} B = {burstiness(lengths):.3f}")

> **Critical check.**
> - Do these values match what you expected from the bar charts in Section 1?
> - Why do `varied` and `wordy` get the same score? (Hint: try dividing the top and bottom of the formula by μ.)
> - B was designed for the **gaps between events in time**. We're applying it to **sentence lengths**.
>   Does "0 means random" still mean anything for sentences? What would a "random" sentence length even be?
>   Borrowing a measure from another field is common, and often useful, but its built-in assumptions come along with it.

## 5. Four novels

Now we put everything into one function that downloads a book and reports on its sentence lengths.
It returns a **dictionary**: a set of labelled values, written with curly brackets `{ }`, which pandas turns into a row of a table.

In [ ]:
def profile(title, book_id):
    lengths = sentence_lengths(strip_gutenberg(get_gutenberg(book_id)))
    return {
        "title": title,
        "sentences": len(lengths),
        "mean": round(statistics.mean(lengths), 1),
        "median": statistics.median(lengths),
        "std dev": round(statistics.pstdev(lengths), 1),
        "burstiness": round(burstiness(lengths), 3),
    }

books = [
    ("Pride and Prejudice", 1342),
    ("Emma", 158),
    ("A Tale of Two Cities", 98),
    ("Frankenstein", 84),
]

results = pd.DataFrame([profile(title, book_id) for title, book_id in books])
results

> **Critical check.**
> - All four novels get a burstiness below 0. What does the sign tell you, if anything, given the discussion in Section 4?
> - The two Austen novels: are they closer to each other than to the other two authors, on each measure?
> - Which differences are big enough to mean something? We have one number per novel, so we have no idea yet how much
>   it would vary between, say, different chapters of the same book. (Section 8 comes back to this.)

### Seeing the rhythm
A number summarises; a picture lets you look. Here are the first 150 sentences of two novels.

In [ ]:
pp = sentence_lengths(strip_gutenberg(get_gutenberg(1342)))
fr = sentence_lengths(strip_gutenberg(get_gutenberg(84)))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, sharey=True)
axes[0].bar(range(150), pp[:150], color="steelblue")
axes[0].set_title("Pride and Prejudice: first 150 sentences")
axes[1].bar(range(150), fr[:150], color="darkorange")
axes[1].set_title("Frankenstein: first 150 sentences")
axes[1].set_xlabel("sentence number")
plt.show()

> **Critical check.** What text are these 150 sentences actually from? (Remember the preface in *Pride and Prejudice*, and
> *Frankenstein* opens with letters.) Try `pp[3000:3150]` instead to look at a passage from the middle of the novel.

## 6. How much does cleaning matter?

Last time we found "sentences" that were really chapter headings, stray numbers, or exclamations like "Oh!".
Let's see what happens to burstiness if we leave out every "sentence" shorter than 3 words.

In [ ]:
def drop_short(lengths, minimum=3):
    return [x for x in lengths if x >= minimum]

print("Pride and Prejudice")
print("  all sentences:  B =", round(burstiness(pp), 3))
print("  3+ words only:  B =", round(burstiness(drop_short(pp)), 3))

> **Critical check.** "Oh!" and "Impossible!" are real sentences, part of Austen's style, not errors.
> A chapter heading is not. A rule like "drop anything under 3 words" can't tell them apart.
> Try changing `minimum` to 1, 5 and 10. How much does B move? How does that compare with the differences *between novels* in Section 5?

## 7. What burstiness can't see: order

Here is something important. Shuffle the sentences of a novel into a random order.
The writing would be nonsense, and its rhythm destroyed. What happens to the burstiness?

In [ ]:
random.seed(1)               # so everyone gets the same "random" shuffle
shuffled = pp.copy()
random.shuffle(shuffled)

print("original order:", round(burstiness(pp), 3))
print("shuffled:      ", round(burstiness(shuffled), 3))

Exactly the same. The standard deviation and the mean only depend on *which* lengths occur, not *in what order*.
So B measures how varied the sentence lengths are, **not** the rhythm of short-after-long that we described in the introduction.

One way to capture order: look at how much the length changes **from one sentence to the next**, on average.

In [ ]:
def average_change(lengths):
    changes = [abs(b - a) for a, b in zip(lengths, lengths[1:])]   # each sentence compared with the next one
    return statistics.mean(changes)

print("original order:", round(average_change(pp), 1), "words")
print("shuffled:      ", round(average_change(shuffled), 1), "words")

> **Critical check.**
> - If Austen deliberately alternated long and short sentences, would the original order give a **bigger** or **smaller** change than the shuffle?
> - Which do you actually see? What might cause neighbouring sentences to be more alike than chance? (Think about stretches of quick dialogue versus long passages of narration.)
> - Try it for Frankenstein (`fr`). Same pattern?
> - Here, the *comparison with a shuffled version* does the real work. A number on its own ("the length changes by about 15 words") means little; a number compared with "what you'd expect if order didn't matter" means much more.

## 8. How reliable is a burstiness score from a short text?

Novels give us thousands of sentences. But people often want to measure a short piece: an essay, a letter, a paragraph
someone suspects was written by AI. Here we take ten random runs of 15 consecutive sentences from *Pride and Prejudice*,
all by the same author, and measure each:

In [ ]:
random.seed(2)
for i in range(10):
    start = random.randrange(len(pp) - 15)
    chunk = pp[start:start + 15]
    print(f"sentences {start:5}-{start + 15:5}:  B = {burstiness(chunk):6.3f}")

> **Critical check.** Same author, same novel. How much does B vary from passage to passage?
> Compare that spread with the differences between the four novels in Section 5.
> What does this tell you about using burstiness to judge who (or what) wrote a single short passage?

## 9. "Burstiness" and generative AI

You may already have met the word *burstiness* in discussions of **detecting AI-written text**: the claim that human writing is
"burstier" than AI writing. Be careful: different people use the same word for **different measurements**.
Some AI-detection tools use it for variation in how *predictable* each sentence is to a language model, not for sentence length at all.
Whenever you see the word, ask: *burstiness of what, measured how, compared with what?*

### Exercise: interrogate an AI
1. Ask an AI assistant: *"What is burstiness in text, and how do I calculate it in Python?"*
2. Paste its code into a new cell. Run it on `metronome`, `varied` and `extreme`. Does it rank them sensibly? Does it agree with our `burstiness` function? If not, is it measuring something different, or is one of them wrong?
3. Ask it where its definition comes from. If it gives a reference, **check that the reference exists** and says what the AI claims it says.
4. Ask it whether human writing is burstier than AI writing. What evidence does it offer? Is that evidence about sentence length, or something else?

### Exercise: measure a passage yourself
Paste a passage between the triple quotes below. It could be a public-domain text from your field, your own writing that you'd be happy to make public, or text generated by an AI.
Don't paste anything unpublished, restricted or personal (see *Your data and Colab* at the top of the warm-up notebook).
Then think about Section 8 before you believe the number.

In [ ]:
passage = """
Paste your passage here. Replace all of this text. Keep the three quotation marks at each end.
"""

lengths = sentence_lengths(passage)
print("sentences:", len(lengths))
print("lengths:  ", lengths)
print("mean:     ", round(statistics.mean(lengths), 1))
print("B:        ", round(burstiness(lengths), 3))

## Summary

| Measure | What it captures | Blind spot |
|---|---|---|
| Mean sentence length | How long sentences are on average | Says nothing about variety |
| Range | The gap between the longest and shortest | Depends on just two sentences, often errors |
| Standard deviation | How varied the lengths are | Grows with sentence length |
| Coefficient of variation, burstiness B | Variety relative to the mean | Ignores order; unstable on short texts; its "0" was designed for another field |
| Average change between neighbours (vs. shuffled) | Something about rhythm and order | Still only about length, not content |

**Next week:** we put several of these measures together, for several authors, and ask whether they can tell writers apart.